# Agentic Factor Research Monitor

This notebook is a **factor research flight recorder** connected to the take-home pipeline:

`part1` audits → `part2` signal hypotheses → `part3` panel → `return_prediction_model` quintile LS + FF4 alpha → **governance agents + monitor**

**Primary path:** real SAS data in the project root. **Fallback:** deterministic synthetic panel if SAS files are missing.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DEMO_ROOT = Path('.').resolve()
if str(DEMO_ROOT) not in sys.path:
    sys.path.insert(0, str(DEMO_ROOT))

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

from data_adapter import load_research_bundle
from agents.orchestrator import ResearchOrchestrator
from cache import persist_research_run

PREFER_REAL = True  # set False to force synthetic demo data

## 1. Load research bundle (real SAS pipeline or synthetic fallback)

In [ ]:
bundle = load_research_bundle(prefer_real=PREFER_REAL)
print(f'Data source: {bundle.data_source}')
print(f'Factors: {list(bundle.summary.columns)}')
print(f'Months: {len(bundle.summary)}')
display(bundle.summary.tail())

if bundle.panel is not None:
    print(f'Panel shape: {bundle.panel.shape}')
    display(bundle.panel.head())

## 2. Backtest metrics (Sharpe, FF4 alpha, turnover)

In [ ]:
cols = ['sharpe', 'cagr', 'max_drawdown', 'hit_rate', 'avg_turnover',
        'ff4_alpha_monthly', 'ff4_alpha_se']
show = [c for c in cols if c in bundle.metrics_df.columns]
display(bundle.metrics_df[show].style.format({
    'sharpe': '{:.2f}', 'cagr': '{:.2%}', 'max_drawdown': '{:.2%}',
    'hit_rate': '{:.2%}', 'avg_turnover': '{:.2%}',
    'ff4_alpha_monthly': '{:.4f}', 'ff4_alpha_se': '{:.4f}',
}))

equity = (1 + bundle.summary.dropna(how='all')).cumprod()
equity.plot(figsize=(11, 4), title='Long-Short Factor Equity Curves')
plt.ylabel('Growth of $1')
plt.show()

## 3. Agentic research governance (multi-agent orchestrator)

In [ ]:
result = ResearchOrchestrator().run(bundle)
run_dir = persist_research_run(result)
print(f'Artifacts saved to: {run_dir}')
print()
display(result.combined_findings)

## 4. Per-factor monitor reports

In [ ]:
for factor, report in result.factor_monitor_reports.items():
    print(f'\n=== {factor} Research Monitor ===')
    display(report)

## 5. Regime diagnostics & rolling Sharpe

In [ ]:
if not bundle.regime_df.empty:
    display(bundle.regime_df.style.format({
        'cagr': '{:.2%}', 'sharpe': '{:.2f}', 'max_drawdown': '{:.2%}', 'hit_rate': '{:.2%}',
    }))

bundle.rolling_sharpe.plot(figsize=(11, 4), title='24-Month Rolling Sharpe')
plt.axhline(0, color='black', linewidth=1)
plt.ylabel('Sharpe')
plt.show()

## 6. Investment research memo

In [ ]:
print(result.memo)